# 이용강도 (Intensity) 모델

# 신용카드 이용강도 기반 이탈 예측 및 수익 최적화
> **목적**: 고객의 결제 금액 및 고액 결제 비중 변화를 분석하여 우량 고객의 이탈 징후를 조기에 포착하고 방어 이익 극대화

### 주요 분석 스토리라인:
1. **가설 설정**: 우량 고객은 완전 이탈 전 결제 금액이 급감하는 '소비 침식' 현상을 보일 것이라 가정
2. **피처 엔지니어링**: 소비기울기, 이용금액 변화율, 3개월 이동평균 등 금액 변동 중심 파생 변수 **산출**
3. **모델링 및 고도화**: 
   - **TimeSeriesSplit(3-Fold)** 적용으로 금융 데이터의 시계열 정합성 및 재현성 **확보**
   - RF, XGB, LGBM, CatBoost 스태킹 후 수치 포착에 최적화된 **LightGBM** 최종 엔진 **채택**
4. **검증 및 진단**: SHAP 기반 변수 기여도 확인 및 Calibration Curve를 통한 예측 확률의 신뢰도 **검증**
5. **결과 도출**: 2019년 1월 이탈 확률값 생성을 통한 마케팅 대상자 명단 **추출**

In [ ]:
# Step 0. 라이브러리 및 한글 폰트 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import polars as pl
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, f1_score, precision_recall_curve, 
    auc, recall_score, precision_score, average_precision_score,
    calibration_curve, confusion_matrix
)
import shap

# 한글 폰트 설정 (Windows: Malgun Gothic)
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

print("STEP 0: 환경 설정 및 라이브러리 로드 완료")

In [ ]:
# STEP 1. 데이터 로드 및 기초 탐색

# 마스터 전처리에서 생성된 최종 데이터셋을 불러옵니다.
df = pl.read_parquet('final_master_dataset.parquet').to_pandas()

print(f"STEP 1: 데이터 로드 완료. Shape: {df.shape}")
print(f"전체 이탈률(Baseline): {df['is_churn'].mean():.2%}")

In [ ]:
# Step 2. Y값 정의 및 변수 및 파생변수 생성

def engineer_intensity_features(df):
    print("STEP 2: 이용강도 중심 파생변수 생성 중...")
    eps = 1e-9
    
    # [소비 변화 지표]
    df['소비기울기'] = df['이용금액_신용_B0M'] - df['이용금액_신용_B1M']
    df['이용금액_변화율'] = df['소비기울기'] / (df['이용금액_신용_B1M'] + eps)
    
    # [집중도 지표]
    df['고액결제_비중'] = df['이용금액_고액_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['온라인_결제비중'] = df['온라인_결제금액_B0M'] / (df['이용금액_신용_B0M'] + eps)
    
    # [효율성 지표]
    df['Ticket_Size'] = df['이용금액_신용_B0M'] / (df['이용건수_신용_B0M'] + eps)
    df['한도사용률'] = df['이용금액_신용_B0M'] / (df['이용한도_B0M'] + eps)
    
    # [장기 지표]
    df['이용금액_3M_Avg'] = df[['이용금액_신용_B0M', '이용금액_신용_B1M', '이용금액_신용_B2M']].mean(axis=1)
    
    return df

df = engineer_intensity_features(df)

In [ ]:
# Step 3. 베이스모델(RF) 선정 및 모델링 (타임스플릿 적용)

print("STEP 3: Random Forest 베이스라인 학습 (TimeSeriesSplit 적용)...")

# 3-Fold 시계열 교차검증 설정
tscv = TimeSeriesSplit(n_splits=3)

# 피처 선정 (원본 노트북 기준 10개 핵심 변수)
intensity_features = [
    '이용금액_신용_B0M', '이용금액_신용_B1M', '소비기울기', 
    '이용금액_변화율', '이용금액_3M_Avg', '고액결제_비중',
    'Ticket_Size', '한도사용률', '온라인_결제비중', '연체여부'
]

X = df[intensity_features]
y = df['is_churn']

rf_base = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

# 폴드별 성능 검증
for i, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    rf_base.fit(X_train, y_train)
    val_probs = rf_base.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_probs)
    print(f"Fold {i+1} Average Precision: {val_ap:.4f}")

In [ ]:
# Step 4. 피처 목록 확정 (SHAP, Permutation, Top-K Lift)

print("STEP 4: 다각도 피처 영향도 분석 (SHAP & Lift)...")

# 1. SHAP Value 시각화
explainer = shap.TreeExplainer(rf_base)
shap_values = explainer.shap_values(X.sample(1000))
shap.summary_plot(shap_values, X.sample(1000), plot_type="bar")

# 2. Top-K Lift 산출 (상위 10% 고객의 이탈률 / 전체 이탈률)
all_probs = rf_base.predict_proba(X)[:, 1]
lift_df = pd.DataFrame({'prob': all_probs, 'actual': y}).sort_values('prob', ascending=False)
top_10_actual = lift_df.head(int(len(lift_df)*0.1))['actual'].mean()
baseline_churn = y.mean()
print(f"Top 10% Lift: {top_10_actual / baseline_churn:.2f}배")

In [ ]:
# Step 5. 모델 고도화 위한 스태킹 진행 (RF, LGBM, CAT, XGB)

print("STEP 5: 4개 엔진 기반 스태킹 앙상블 수행...")

stack_models = {
    'RF': rf_base,
    'XGB': XGBClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'LGBM': LGBMClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'CAT': CatBoostClassifier(iterations=200, auto_class_weights='Balanced', verbose=0)
}

oof_preds = pd.DataFrame(index=df.index)

for name, model in stack_models.items():
    print(f"  - {name} 학습 중...")
    model.fit(X, y)
    oof_preds[name] = model.predict_proba(X)[:, 1]

# Meta-Learner: ElasticNet (Logistic Regression with L1/L2)
meta_learner = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5)
meta_learner.fit(oof_preds, y)
final_stacking_probs = meta_learner.predict_proba(oof_preds)[:, 1]

In [ ]:
# Step 6. 주제별 엔진 선정 (이용강도 - LightGBM 선정)

print("STEP 6: 이용강도 최종 엔진(LightGBM) 모델링...")

final_lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=5,
    random_state=42
)

final_lgbm.fit(X, y)
final_intensity_probs = final_lgbm.predict_proba(X)[:, 1]

In [ ]:
# Step 7. 2019년 1월 이탈자 예측 결과 저장

print("STEP 7: 마케팅 통합을 위한 예측 결과 저장 (2019.01)...")

# 예측 확률과 실제값을 회원번호와 매칭
result_df = df[['발급회원번호', 'is_churn']].copy()
result_df['intensity_prob'] = final_intensity_probs

# 위험도 세그먼트(Tier) 생성
result_df['Risk_Tier'] = pd.qcut(result_df['intensity_prob'], 4, labels=['Safe', 'Tier3', 'Tier2', 'Tier1'])

# 최종 저장
result_df.to_csv('intensity_churn_full_results.csv', index=False, encoding='utf-8-sig')
print("intensity_churn_full_results.csv 저장 완료.")

In [ ]:
# Step 8. 모델 검증 (PSI, Calibration Curve)

print("STEP 8: 최종 모델 안정성(PSI) 및 신뢰도(Calibration) 진단...")

# 1. Calibration Curve
prob_true, prob_pred = calibration_curve(y, final_intensity_probs, n_bins=10)
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Intensity LGBM')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Calibration Plot (Model Reliability)')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.show()

# 2. PSI (Population Stability Index) - 원본 커스텀 로직 반영
def calculate_psi(expected, actual, buckets=10):
    def scale_range(input_array):
        return (input_array - input_array.min()) / (input_array.max() - input_array.min())
    
    e_counts = np.histogram(scale_range(expected), buckets)[0] / len(expected)
    a_counts = np.histogram(scale_range(actual), buckets)[0] / len(actual)
    
    def psi(e, a):
        if a == 0: a = 0.0001
        if e == 0: e = 0.0001
        return (e - a) * np.log(e / a)
    
    return sum([psi(e_counts[i], a_counts[i]) for i in range(buckets)])

psi_val = calculate_psi(y[:5000], y[5000:10000]) # 샘플 구간 비교
print(f"PSI Score: {psi_val:.4f}")